# Tahap 6: Model Evaluation

Notebook ini merepresentasikan blok abu-abu **Model Evaluation** pada diagram metodologi penelitian, yang mengevaluasi dan membandingkan performa ketiga metode (**LDA**, **Top2Vec**, dan **BERTopic**) melalui 3 sub-tahapan:
1. **Coherence Score ($C_v$)** $\rightarrow$ Mengukur koherensi semantik kata-kata di dalam satu topik.
2. **Topic Diversity** $\rightarrow$ Mengukur proporsi keunikan kata kunci antar topik (seberapa terpisah topik satu sama lain).
3. **Topic Visualization** $\rightarrow$ Visualisasi perbandingan performa, kata kunci utama per topik, dan pemetaan ruang topik 2D.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import umap

os.makedirs('evaluation_outputs', exist_ok=True)
sns.set_theme(style='whitegrid')

## 6.0 Memuat Hasil Pelatihan Model & Korpus Token

In [ ]:
with open('representations/bow_representation.pkl', 'rb') as f:
    bow_data = pickle.load(f)

tokenized_docs = bow_data['tokenized_docs']
full_dictionary = Dictionary(tokenized_docs)

with open('models/training_results.pkl', 'rb') as f:
    results = pickle.load(f)

models_list = ['LDA', 'Top2Vec', 'BERTopic']
print('Hasil model yang dievaluasi:', models_list)

## 6.1 Evaluasi 1: Coherence Score ($C_v$)
Menghitung nilai **Coherence Score ($C_v$)** untuk masing-masing model menggunakan `CoherenceModel` dari Gensim. Semakin tinggi nilai $C_v$ (mendekati 1.0), semakin koheren dan mudah diinterpretasikan topik yang dihasilkan.

In [ ]:
def compute_coherence_cv(topics, tokenized_texts, dictionary):
    # Pastikan hanya kata yang ada di kamus yang dihitung
    valid_topics = [
        [w for w in topic if w in dictionary.token2id]
        for topic in topics
    ]
    valid_topics = [t for t in valid_topics if len(t) >= 2]
    cm = CoherenceModel(
        topics=valid_topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

coherence_scores = {}
for m_name in models_list:
    score = compute_coherence_cv(results[m_name]['topics'], tokenized_docs, full_dictionary)
    coherence_scores[m_name] = score
    print(f'Coherence Score (Cv) - {m_name:8s}: {score:.4f}')

## 6.2 Evaluasi 2: Topic Diversity
**Topic Diversity** mengukur persentase kata unik di dalam Top-$N$ kata pada seluruh topik:
$$\text{Topic Diversity} = \frac{|\text{Kata Unik pada Seluruh Topik}|}{K \times N}$$
Nilai mendekati `1.0` menunjukkan setiap topik sangat spesifik dan tidak saling tumpang tindih (*redundant*).

In [ ]:
def compute_topic_diversity(topics, top_n=10):
    all_words = [w for topic in topics for w in topic[:top_n]]
    if len(all_words) == 0:
        return 0.0
    unique_words = set(all_words)
    return len(unique_words) / len(all_words)

diversity_scores = {}
for m_name in models_list:
    div = compute_topic_diversity(results[m_name]['topics'], top_n=results['top_n_words'])
    diversity_scores[m_name] = div
    print(f'Topic Diversity      - {m_name:8s}: {div:.4f}')

# Menyusun Tabel Perbandingan Performa Lengkap
df_eval = pd.DataFrame({
    'Metode': models_list,
    'Text Representation': ['Bag-of-Words', 'Semantic Vector Space', 'Transformer Embedding'],
    'Coherence Score (Cv)': [coherence_scores[m] for m in models_list],
    'Topic Diversity': [diversity_scores[m] for m in models_list],
    'Waktu Training (detik)': [results[m]['training_time'] for m in models_list]
})

df_eval.to_csv('evaluation_outputs/tabel_perbandingan_performa.csv', index=False)
df_eval

## 6.3 Evaluasi 3: Topic Visualization
### A. Visualisasi Perbandingan Metrik Performa (Coherence Score & Topic Diversity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = ['#4C72B0', '#DD8452', '#55A868']

# Plot Coherence Score
sns.barplot(data=df_eval, x='Metode', y='Coherence Score (Cv)', palette=palette, ax=axes[0])
axes[0].set_title('Perbandingan Coherence Score (Cv)', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, max(df_eval['Coherence Score (Cv)']) * 1.2)
for i, v in enumerate(df_eval['Coherence Score (Cv)']):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# Plot Topic Diversity
sns.barplot(data=df_eval, x='Metode', y='Topic Diversity', palette=palette, ax=axes[1])
axes[1].set_title('Perbandingan Topic Diversity', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1.15)
for i, v in enumerate(df_eval['Topic Diversity']):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_outputs/perbandingan_metrik.png', dpi=300)
plt.show()

### B. Visualisasi Kata Kunci Utama (*Top Words*) per Topik pada Setiap Metode

In [ ]:
def plot_model_top_words(model_name, topic_weights, color_map='Blues_r'):
    num_t = len(topic_weights)
    fig, axes = plt.subplots(1, num_t, figsize=(4 * num_t, 5), sharex=False)
    if num_t == 1:
        axes = [axes]
    
    for idx, words_scores in enumerate(topic_weights):
        words = [w for w, _ in words_scores[:8]][::-1]
        scores = [s for _, s in words_scores[:8]][::-1]
        axes[idx].barh(words, scores, color=sns.color_palette(color_map, len(words)))
        axes[idx].set_title(f'Topik {idx + 1}', fontweight='bold')
        axes[idx].tick_params(axis='y', labelsize=10)
    
    fig.suptitle(f'Top Kata Kunci per Topik - Metode {model_name}', fontsize=15, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig(f'evaluation_outputs/top_words_{model_name.lower()}.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_model_top_words('LDA', results['LDA']['topic_weights'], 'Blues_r')
plot_model_top_words('Top2Vec', results['Top2Vec']['topic_weights'], 'Oranges_r')
plot_model_top_words('BERTopic', results['BERTopic']['topic_weights'], 'Greens_r')

### C. Visualisasi Sebaran Klaster Topik Dokumen dalam Ruang 2D (UMAP Projection)

In [ ]:
transformer_embeddings = np.load('representations/transformer_embeddings.npy')
umap_2d = umap.UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
coords_2d = umap_2d.fit_transform(transformer_embeddings)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, m_name in enumerate(models_list):
    doc_labels = [f'Topik {t+1}' if t != -1 else 'Outlier' for t in results[m_name]['doc_topics']]
    sns.scatterplot(
        x=coords_2d[:, 0],
        y=coords_2d[:, 1],
        hue=doc_labels,
        palette='tab10',
        alpha=0.7,
        s=25,
        ax=axes[idx]
    )
    axes[idx].set_title(f'Distribusi Topik Dokumen - {m_name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('UMAP Dimensi 1')
    axes[idx].set_ylabel('UMAP Dimensi 2')
    axes[idx].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.savefig('evaluation_outputs/visualisasi_distribusi_2d.png', dpi=300)
plt.show()